# पाठ 11 - Model Context Protocol (MCP)

The **Model Context Protocol (MCP)** एक खुला मानक हो जसले एजेन्टहरूलाई रनटाइममा उपकरणहरू, स्रोतहरू, र डाटा स्रोतहरू गतिशील रूपमा पत्ता लगाउन र प्रयोग गर्न सक्षम बनाउँछ। एउटा एजेन्टमा उपकरणहरू हार्डकोड गर्ने सट्टा, MCP ले एजेन्टहरूलाई आवश्यक अनुसार क्षमताहरू खुला गर्ने बाह्य सर्भरहरूसँग जडान हुन अनुमति दिन्छ।

यस पाठमा, तपाईंले सिक्नु हुनेछ:
- MCP के हो र एजेन्ट प्रणालीहरूका लागि यो किन महत्त्वपूर्ण छ
- MCP क्लाइन्ट-सर्भर वास्तुकला कसरी काम गर्छ
- MCP-शैलीको उपकरण खोज प्रयोग गर्ने एजेन्टहरू कसरी निर्माण गर्ने


## Setup

**पूर्वापेक्षाहरू:**
- Azure AI Foundry परियोजना जसमा मोडेल परिनियोजित गरिएको छ
- प्रमाणीकरणका लागि `az login` चलाउनुहोस्


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## मोडेल सन्दर्भ प्रोटोकल (MCP) भनेको के हो?

MCP ले कृत्रिम बुद्धिमत्ता (AI) एजेन्टहरूले बाह्य उपकरण र डेटा स्रोतहरू पत्ता लगाएर र तिनीहरूसँग अन्तरक्रिया गरेर प्रयोग गर्ने मानक तरिका परिभाषित गर्छ:

- **MCP Server**: मानक प्रोटोकलमार्फत उपकरणहरू, स्रोतहरू, र प्रम्प्टहरू उपलब्ध गराउँछ
- **MCP Client**: सर्वरहरूमा जडान हुने र उपलब्ध क्षमता पत्ता लगाउने एजेन्ट रनटाइम
- **Dynamic Discovery**: एजेन्टहरूलाई हार्डकोड गरिएको उपकरणहरू आवश्यक पर्दैन — तिनीहरूले रनटाइममा उपलब्ध के छ पत्ता लगाउँछन्

यो त्यस्ता विस्तारयोग्य एजेन्ट प्रणालीहरू निर्माण गर्न शक्तिशाली हो जहाँ नयाँ क्षमताहरू एजेन्टको कोड परिवर्तन गर्नु पर्ने बिना थप्न सकिन्छ।


## MCP कसरी काम गर्छ

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. एजेन्ट (MCP client) एउटा MCP server सँग जडान हुन्छ
2. सर्भर उपलब्ध उपकरणहरू र तिनीहरूको स्कीमाहरूको सूचीसहित प्रतिक्रिया दिन्छ
3. एजेन्टले त्यसपछि आफ्नो तर्क प्रक्रियाको क्रममा फेला परेका कुनै पनि उपकरणहरूलाई आह्वान गर्न सक्छ
4. नतिजाहरू उही प्रोटोकलमार्फत फर्केर आउँछन्


## MCP उपकरण पत्ता लगाउने अनुकरण

वास्तविक MCP सर्भरले चलिरहेको सर्भर प्रक्रिया आवश्यक पर्ने हुँदा, हामी `@tool` फङ्सनहरू प्रयोग गरेर त्यो ढाँचा प्रदर्शन गर्नेछौं, जसले MCP-सँग जडित आवास सेवाले प्रदान गर्ने कुराहरू अनुकरण गर्छ।

उत्पादनमा, यी उपकरणहरू स्थानीय रूपमा परिभाषित गरिने सट्टा MCP सर्भरबाट गतिशील रूपमा पत्ता लगाइने थिए।


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP-शैलीका उपकरणहरूसँग एजेṇ्ट निर्माण


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## उत्पादनमा MCP

उत्पादन वातावरणमा, MCP ले शक्तिशाली ढाँचाहरू सक्षम बनाउँछ:

- **डायनामिक उपकरण पत्ता लगाउने**: एजेन्टहरूले MCP सर्भरसँग जडान हुन्छन् र रनटाइममा उपकरणहरू पत्ता लगाउँछन्
- **विच्छेदित वास्तुकला**: उपकरण प्रदायकहरू एजेन्टबाट स्वतन्त्र रूपमा अद्यावधिक गर्न सक्छन्
- **संगठनहरूबीचको साझेदारी**: टिमहरूले MCP सर्भरहरू मार्फत क्षमताहरू उपलब्ध गराउन सक्छन् जुन कुनै पनि एजेन्टले प्रयोग गर्न सक्छ
- **Microsoft Agent Framework समर्थन**: MAF मा `mcp` इन्टिग्रेसन मार्फत निहित MCP क्लाइन्ट समर्थन छ

वास्तविक MCP सर्भरलाई MAF सँग प्रयोग गर्न, तपाईं `hosted_mcp_tool()` वा MCP क्लाइन्ट इन्टिग्रेसन मार्फत जडान गर्नुहुनेछ।

**थप जान्नुहोस्:**
- [MCP विशिष्टकरण](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP समर्थन](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## सारांश

यस पाठमा, तपाईंले सिक्नुभयो:
- **MCP** एजेन्टहरू र उपकरण प्रदायकहरू बीच गतिशील उपकरण पत्ता लगाउने लागि खुला मानक हो
- **क्लाइएन्ट-सर्भर वास्तुकला** ले एजेन्टहरूलाई रनटाइममा क्षमताहरू पत्ता लगाउन अनुमति दिन्छ
- MCP ले **विस्तारयोग्य र पृथक एजेन्ट प्रणालीहरू** सक्षम पार्छ जहाँ उपकरणहरू कोड परिवर्तन बिना थप्न सकिन्छ
- Microsoft Agent Framework उत्पादन प्रयोगका लागि **अन्तर्निहित MCP समर्थन** प्रदान गर्छ


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
अस्वीकरण:
यो दस्तावेज AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सटीकता सुनिश्चित गर्न प्रयास गर्छौं, तर कृपया ध्यान दिनुहोस् कि स्वचालित अनुवादमा त्रुटि वा अशुद्धता हुन सक्छ। मूल भाषामा रहेको दस्तावेजलाई अधिकारिक स्रोतको रूपमा मान्नुपर्छ। महत्त्वपूर्ण जानकारीका लागि पेशेवर मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न हुने कुनै पनि गलतफहमी वा गलत व्याख्याका लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
